# Hospital master data quality
Run from the project root or this notebook directory. The reusable checks are in `scripts/check_master.py`. Passing integrity checks does not certify full NMPA coverage or every entity identity.

In [ ]:
from pathlib import Path
import json, runpy, sqlite3
root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = root.parent
checks = runpy.run_path(str(root / 'scripts/check_master.py'))
checks['check'](root / 'data/master')

In [ ]:
db = sqlite3.connect(f'file:{root / "data/master/hospital_master.sqlite"}?mode=ro', uri=True)
db.execute('SELECT verification_status, COUNT(*), SUM(ctg_trial_count) FROM institutions GROUP BY verification_status').fetchall()

In [ ]:
db.execute('SELECT i.canonical_name, i.ctg_trial_count, i.nmpa_trial_count, p.review_status FROM review_priority p JOIN institutions i USING(hospital_id) ORDER BY p.priority_rank LIMIT 20').fetchall()

In [ ]:
print(json.dumps(json.loads((root / 'data/master/quality_summary.json').read_text()), indent=2, ensure_ascii=False))
db.close()